# QLoRA training of Qwen3-4B (Milestone 3)

Trains the fine-tuned Qwen3-4B under `docs/milestone-2/EVAL_PROTOCOL.md`, Amendment 1, on **train data only**: the training file built from the train split (checked against `docs/milestone-3/train-v1.manifest.json`) under the epoch budget logged in `docs/milestone-3/VALIDATION_LOG.md` (every non-probe example, probe examples at rate 0.9707; checked against `docs/milestone-3/train-v1-budget.json`): 16,869 examples, 1,055 optimiser steps of 16, on two T4s (data parallel). Checkpoints at steps 264, 528, 792 and 1,055 (25%, 50%, 75%, 100%) are the four selection candidates (`checkpoint-<step>`); every 66 steps in between a resume-only checkpoint (`resume-<step>`, never evaluated) is written, and only the newest one is kept. Each holds the LoRA weights, the adapter for evaluation, the optimiser, scheduler and fp16 loss-scaler states, every process's random number generator states and the step reached, with every file's sha256 in its `state.json`.

Two sessions of about 7 hours each (Kaggle's limit is 12):

- **Session 1** (`SESSION = 1`): steps 1–528, writes `checkpoint-0264` and `checkpoint-0528`.
- **Session 2** (`SESSION = 2`): resumes from `checkpoint-0528` and runs steps 529–1,055, writes `checkpoint-0792` and `checkpoint-1055`.

**Setup, session 1:** Accelerator *GPU T4 x2*, Internet *on*, attach the private Kaggle Dataset with the Milestone 3 bundle, set `SESSION = 1`, then *Save Version → Save & Run All*. When it finishes, note the `state.json` sha256 of `checkpoint-0528` printed at the end (it is also in `run-steps-0000-0528.json`) and download `qlora-train-session-1.tar.gz` from the Output tab.

**Carrying checkpoint-0528 into session 2:**

1. Open the finished session-1 notebook and choose *Copy & Edit*: the copy is a separate notebook whose inputs can include session 1's output.
2. In the copy: *Add Input → Your Work → Notebooks*, select the session-1 notebook (its saved version with the finished run). Its output appears under `/kaggle/input/<notebook name>/`, `checkpoint-0528` included. Keep the bundle dataset attached.
   - If Kaggle does not offer the notebook's output: upload `checkpoint-0528` from the session-1 archive as a private Kaggle Dataset and attach that instead; the notebook finds it in either place.
   - Session 1's output also holds that session's copy of the code (`hetren/`). The notebook ignores it and takes its code only from the attached bundle. Set `BUNDLE_COMMIT` to the bundle's commit so that the notebook stops if it finds a different one.
3. Set `SESSION = 2` and check that `RESUME_SHA256` is the sha256 noted above (`613ac6d5…8315` for `checkpoint-0528`, logged in the validation log), then *Save Version → Save & Run All*. The run stops before training unless `state.json` has exactly that sha256 and every checkpoint file matches the hash `state.json` lists for it.
4. Before the long run, a **one-step resume check** loads the checkpoint in a fresh process and runs one optimiser step, saving nothing. If the resume cannot work, the notebook stops there, about 10 minutes in.

**After a crash or a non-finite loss**, resume from the latest saved checkpoint. Its name and `state.json` sha256 are the last `"checkpoint"` entry in `train-log.jsonl` (written as each checkpoint is saved). If Kaggle kept no output from the failed version, resume from the newest checkpoint in an output you do have, for example the previous session's.

- Attach the output that holds it, as for session 2.
- Set `RESUME_FROM` to its name (for example `resume-0396`) and `RESUME_SHA256` to its sha256.
- Keep the session's `UNTIL_CHECKPOINT`: 2 to finish session 1's part, 4 for session 2's.
- **After a crash:** keep `LR_SCALE = 1.0`.
- **After a non-finite loss** (exit code 3, outcome `stopped: non-finite loss ...`): set `LR_SCALE = 0.5` (Amendment 1). A second non-finite loss stops training.
- **Before step 66:** there is no checkpoint, so restart session 1 with `LR_SCALE = 0.5`.

Every run and every resume, including failed and stopped ones, is logged in the validation log.

In [ ]:
SESSION = 1  # 1 or 2
RESUME_SHA256 = {1: "", 2: "613ac6d5e4595a56752804a93701dadb538b06d9dd30b462feaccfb377988315"}[SESSION]  # checkpoint-0528's state.json (validation log)
RESUME_FROM = {1: None, 2: "checkpoint-0528"}[SESSION]  # after a crash or non-finite loss: the latest checkpoint in train-log.jsonl
UNTIL_CHECKPOINT = {1: 2, 2: 4}[SESSION]
LR_SCALE = 1.0  # 0.5 only after a non-finite loss (Amendment 1); a crash resumes at 1.0
PROBE_RATE = "0.9707"  # the epoch budget in docs/milestone-3/VALIDATION_LOG.md
BUNDLE_COMMIT = ""  # optional: the bundle's commit (full sha); the notebook stops if the bundle found is another

import glob, hashlib, json, os, shutil, subprocess, sys, tarfile

WORK = "/kaggle/working/hetren"
DATA = "/kaggle/working/train-v1"
OUT = f"/kaggle/working/qlora-train-session-{SESSION}"


def earlier_session(path):  # a copy inside an attached earlier session's output, not the bundle itself
    return bool(glob.glob(f"{path.split('/hetren/')[0]}/qlora-train-session-*")) if "/hetren/" in path else False


if not os.path.exists(WORK):
    unpacked = [p for p in glob.glob("/kaggle/input/**/BUNDLE.json", recursive=True) if not earlier_session(p)]
    packed = [p for p in glob.glob("/kaggle/input/**/hetren-*bundle.tar.gz", recursive=True) if not earlier_session(p)]

    if len(unpacked) > 1 or (not unpacked and len(packed) != 1):
        raise SystemExit(f"STOP: attach exactly one bundle; found {unpacked or packed}")

    if unpacked:  # Kaggle may unpack archives on upload
        shutil.copytree(os.path.dirname(unpacked[0]), WORK)
    else:
        os.makedirs(WORK)
        tarfile.open(packed[0]).extractall(WORK)

    print("bundle from", unpacked[0] if unpacked else packed[0])

COMMIT = json.load(open(f"{WORK}/BUNDLE.json"))["commit"]
print("bundle commit:", COMMIT)

if BUNDLE_COMMIT and COMMIT != BUNDLE_COMMIT.strip():
    raise SystemExit(f"STOP: the bundle is commit {COMMIT}, not BUNDLE_COMMIT {BUNDLE_COMMIT!r}")

os.chdir(WORK)

In [ ]:
# The versions the throughput pilot fixed (docs/milestone-3/VALIDATION_LOG.md).
!pip install -q jsonschema==4.26.0 openai==3.42.0 openai-agents==0.24.0 tiktoken==0.14.0 transformers==4.56.2 accelerate==1.10.1 peft==0.17.1 bitsandbytes==0.50.2
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!python -c "import torch, bitsandbytes; n = torch.cuda.device_count(); print(n, 'GPUs:', [torch.cuda.get_device_name(i) for i in range(n)]); assert n == 2"

## Training data (train split only), checked against the committed manifest and budget selection

In [ ]:
subprocess.run([sys.executable, "-m", "packages.support_evaluator.training_data", "--out", DATA,
                "--check", "docs/milestone-3/train-v1.manifest.json",
                "--probe-rate", PROBE_RATE, "--check-budget", "docs/milestone-3/train-v1-budget.json"], check=True)

## Checkpoint to resume from (session 2), checked by hash

In [ ]:
resume = ["--lr-scale", str(LR_SCALE)]

if RESUME_FROM is not None:
    found = sorted(glob.glob(f"/kaggle/input/**/{RESUME_FROM}/state.json", recursive=True))

    if len(found) != 1:
        raise SystemExit(f"STOP: expected one {RESUME_FROM} among the inputs, found {found}")

    digest = hashlib.sha256(open(found[0], "rb").read()).hexdigest()

    if digest != RESUME_SHA256.strip():
        raise SystemExit(f"STOP: {found[0]} has sha256 {digest}, not RESUME_SHA256 {RESUME_SHA256!r}")

    resume += ["--resume", os.path.dirname(found[0]), "--resume-sha256", digest]
    print("resuming from", os.path.dirname(found[0]), "state.json sha256", digest)

## GPU smoke check of checkpointing (session 1 only, about 10 minutes)

Two steps with a checkpoint after each, then a resume from the first checkpoint that repeats the second step, in a separate folder (`/kaggle/working/qlora-smoke`). This checks the save-and-resume path on the GPU: the PEFT adapter, the fp16 loss scaler and the command itself. It does not check a resumed paged optimiser state, because the loss scaler skips step 1 while it calibrates, so `checkpoint-0001` has no optimiser state yet; the one-step resume check before session 2's run covers that. The real run below starts from scratch and is not affected. Whether the two step-2 weights are bit-identical is printed for information only (GPU arithmetic need not be deterministic).

In [ ]:
if SESSION == 1 and RESUME_FROM is None and LR_SCALE == 1.0:
    SMOKE = "/kaggle/working/qlora-smoke"
    base = ["torchrun", "--nproc_per_node", "2", "-m", "packages.hetren_baselines.qlora", "train",
            "--data", f"{DATA}/train-budget.jsonl", "--bundle-commit", COMMIT, "--smoke-steps", "2"]
    subprocess.run([*base, "--out", f"{SMOKE}/straight"], check=True)
    first = f"{SMOKE}/straight/checkpoint-0001"
    digest = hashlib.sha256(open(f"{first}/state.json", "rb").read()).hexdigest()
    subprocess.run([*base, "--out", f"{SMOKE}/resumed", "--resume", first, "--resume-sha256", digest], check=True)
    same = [json.load(open(f"{SMOKE}/{run}/checkpoint-0002/state.json"))["files"]["trainable.pt"]
            for run in ("straight", "resumed")]
    print("smoke check passed; step-2 weights bit-identical:", same[0] == same[1])

## One-step resume check (whenever resuming, about 5 minutes)

Loads the checkpoint in a fresh process, as the real run will, restores the paged 8-bit optimiser state into paged memory, and runs one optimiser step, in a separate folder (`/kaggle/working/qlora-resume-check`). Nothing is saved, and the real run below resumes from the same checkpoint, so the check does not affect it. If it fails, the notebook stops here and archives the check's output.

In [ ]:
CHECK = "/kaggle/working/qlora-resume-check"

if RESUME_FROM is not None:
    check = ["torchrun", "--nproc_per_node", "2", "-m", "packages.hetren_baselines.qlora", "train",
             "--data", f"{DATA}/train-budget.jsonl", "--out", CHECK, "--resume-check", "--bundle-commit", COMMIT, *resume]
    check_exit = subprocess.run(check).returncode

    if check_exit != 0:
        with tarfile.open(f"/kaggle/working/qlora-resume-check-session-{SESSION}.tar.gz", "w:gz") as results:
            results.add(CHECK, arcname=os.path.basename(CHECK))

        raise SystemExit(f"STOP: the one-step resume check exited with {check_exit}; log this attempt as failed")

    steps = [json.loads(line) for line in open(f"{CHECK}/train-log.jsonl") if '"loss"' in line]
    print("resume check passed:", steps)

## Training (two T4s, data parallel)

In [ ]:
command = ["torchrun", "--nproc_per_node", "2", "-m", "packages.hetren_baselines.qlora", "train",
           "--data", f"{DATA}/train-budget.jsonl", "--out", OUT, "--until-checkpoint", str(UNTIL_CHECKPOINT),
           "--bundle-commit", COMMIT, *resume]
EXIT = subprocess.run(command).returncode
print("exit", EXIT)

## Outputs (checkpoints stay in the notebook output for session 2; the archive is for the repository's log)

In [ ]:
with tarfile.open(f"/kaggle/working/qlora-train-session-{SESSION}.tar.gz", "w:gz") as results:
    results.add(OUT, arcname=os.path.basename(OUT))

    for path in glob.glob("/kaggle/working/qlora-smoke/*/run-*.json"):  # the smoke check's records (session 1)
        results.add(path, arcname=os.path.relpath(path, "/kaggle/working"))

    if os.path.exists(CHECK):  # the one-step resume check's record and log
        results.add(CHECK, arcname=os.path.basename(CHECK))

for path in sorted(glob.glob(f"{OUT}/run-*.json")):
    run = json.load(open(path))
    print(os.path.basename(path), run["outcome"], "steps", run["start_step"], "to", run.get("steps_reached"))

    for name, digest in run["checkpoints_written"].items():
        print(f"  {name}: state.json sha256 {digest}")

if EXIT != 0:
    raise SystemExit(f"training exited with {EXIT}; log this run as stopped or failed")